# Assessment Notebook

## Problem 1: Representing SHA-256 Data

This problem involves: research implementing SHA-256, investigate how the input, output, and intermediate data can be represented in Python.

### 32-bit words

SHA-256 operates on groups of 32 bits called words. Each word is an unsigned integer with a range from (0) to (2^{32} - 1). A 32-bit word can be represented using eight hexadecimal digits, as each hexadecimal digit represents four bits.

For example:

In [1]:
import numpy as np

word = np.uint32(0x12345678)

print(word)
print(word.dtype)

305419896
uint32


`numpy.uint32` is appropriate because it explicitly represents an unsigned 32-bit integer, matching the word size used by SHA-256. This is preferable to a floating-point type because SHA-256 relies on integer and bitwise operations such as XOR, AND and bit shifting.

The fixed 32-bit representation is also important because SHA-256 uses arithmetic modulo (2^{32}). For example:

In [2]:
import numpy as np

word = np.uint32(0xFFFFFFFF)
one = np.uint32(1)

result = np.uint32((int(word) + int(one)) & 0xFFFFFFFF)

print(f"{result:08x}")

00000000


This results in `00000000`, demonstrating the 32-bit wraparound from (2^{32} - 1 + 1).

Hexadecimal is mainly used as a convenient way of displaying the word; the underlying value is still a 32-bit unsigned integer.

### Sequences of 32-bit words

A SHA-256 word is exactly 32 bits, so it can represent an unsigned integer in the range:

$$
0 \leq x \leq 2^{32} - 1
$$

In Python, a sequence of these words can be represented using a NumPy array with the `numpy.uint32` data type. This ensures that each element occupies exactly 32 bits and that arithmetic and bitwise operations behave similarly to the operations required by SHA-256.

For example:

In [3]:
import numpy as np

words = np.array([
    0x12345678,
    0xABCDEF01,
    0x00000000,
    0xFFFFFFFF
], dtype=np.uint32)

print(words)
print(words.dtype)

[ 305419896 2882400001          0 4294967295]
uint32


This produces an array where every element is explicitly represented as a 32-bit unsigned integer.

The hexadecimal notation is useful because eight hexadecimal digits correspond to 32 bits:
$$
0x12345678=00010010 00110100 01010110 01111000
$$
A NumPy `uint32` is particularly suitable for SHA-256 because the algorithm requires operations such as addition modulo $2^{32}$, bitwise AND, OR, XOR, and rotations.

### Input Messages

SHA-256 accepts an input message of arbitrary length. The message is ultimately processed as a sequence of bytes, so bytes is an appropriate representation in Python.

A text message can initially be represented as a Python `str`, but it must be converted into bytes before being passed to SHA-256. This can be done using the `.encode()` method. UTF-8 encoding is suitable for representing text, with each character being converted into one or more bytes.

For example, the message `"abc"` can be represented as follows:

In [4]:
# Example input message 
message = "abc" 

# Convert the message into UTF-8 bytes 
message_bytes = message.encode("utf-8") 

print("Message:", message) 
print("Bytes:", message_bytes) 
print("Length:", len(message_bytes))

Message: abc
Bytes: b'abc'
Length: 3


The resulting `bytes` object contains the three byte values corresponding to the characters `a`, `b`, and `c`. These bytes form the input data that can then be processed by the SHA-256 algorithm.

The message `"abc"` is also used as an example in the SHA-256 specification ([FIPS 180-4, Section 5.1.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=18)).

### 512-bit message blocks

SHA-256 processes messages in blocks of 512 bits, equivalent to 64 bytes. According to ([NIST FIPS 180-4, Section 5.2.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=17)), the padded message is divided into 512-bit blocks, with each block consisting of sixteen 32-bit words.

In Python, a block can be represented using a `bytes` object containing exactly 64 bytes. This is appropriate because each byte represents eight bits, making it straightforward to store and manipulate the binary input. The bytes can then be converted into a NumPy array of sixteen `uint32` values, matching the word structure used by SHA-256.

For example:

In [9]:
import numpy as np

# Create an example 512-bit message block
block = bytes(range(64))

print("Block size in bytes:", len(block))
print("Block size in bits:", len(block) * 8)
print("First 8 bytes:", block[:8])

Block size in bytes: 64
Block size in bits: 512
First 8 bytes: b'\x00\x01\x02\x03\x04\x05\x06\x07'


The block contains 64 bytes, giving a total of 512 bits. The `bytes` representation is useful for storing the original message data, while NumPy can be used to represent the sixteen 32-bit words required for processing.

Each group of four bytes must be interpreted in big-endian order, meaning that the most significant byte comes first. This follows the convention specified in FIPS 180-4.

In [10]:
# Convert each group of four bytes into a 32-bit word
words = np.array([
    int.from_bytes(block[i:i + 4], byteorder="big")
    for i in range(0, 64, 4)
], dtype=np.uint32)

print("Number of words:", len(words))
print("Data type:", words.dtype)
print("First four words:", [f"{word:08x}" for word in words[:4]])

Number of words: 16
Data type: uint32
First four words: ['00010203', '04050607', '08090a0b', '0c0d0e0f']


This produces sixteen unsigned 32-bit integers. For this example, the first four words are 00010203, 04050607, 08090a0b, and 0c0d0e0f.

Using `numpy.uint32` is appropriate because SHA-256 performs its operations on 32-bit words, including bitwise operations and addition modulo $(2^{32})$. The array representation therefore provides a convenient way to organise the message block for subsequent stages of the algorithm.

The original bytes object and the array of words serve different purposes: bytes preserves the input as a sequence of eight-bit values, while the NumPy array provides the sixteen 32-bit words that the SHA-256 compression function uses.

### The final 256-bit hash value

SHA-256 produces a fixed-length message digest of 256 bits, regardless of the length of the input message. The digest is the final output of the hashing algorithm and provides a compact representation of the input data. NIST specifies the digest size and algorithm in the Secure Hash Standard ([FIPS 180-4, Section 6.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=27)).

Since one byte contains eight bits, the digest occupies:

$$
\frac{256\text{ bits}}{8\text{ bits per byte}} = 32\text{ bytes}
$$

A SHA-256 digest is commonly displayed in hexadecimal form. Each hexadecimal digit represents four bits, so the complete digest contains:

$$
\frac{256\text{ bits}}{4\text{ bits per hexadecimal digit}} = 64\text{ hexadecimal digits}
$$

In Python, the digest can be represented as a bytes object containing 32 bytes and converted into a hexadecimal string using the .hex() method. This is appropriate because bytes stores the digest as a sequence of eight-bit values, while hexadecimal provides a readable representation of the same binary data. Each byte requires two hexadecimal digits, so the hexadecimal representation is twice as long as the byte sequence.

For example, the SHA-256 test message "abc" produces the following digest. This value is defined by the SHA-256 specification and is a 256-bit output, as described in ([NIST FIPS 180-4, Section 6.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=27)).

In [1]:
# Example final SHA-256 hash value for "abc"
hash_value = bytes.fromhex(
    "ba7816bf8f01cfea414140de5dae2223"
    "b00361a396177a9cb410ff61f20015ad"
)

print("Hash:", hash_value.hex())
print("Size in bytes:", len(hash_value))
print("Size in bits:", len(hash_value) * 8)
print("Hexadecimal digits:", len(hash_value.hex()))

Hash: ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
Size in bytes: 32
Size in bits: 256
Hexadecimal digits: 64


The code uses `bytes.fromhex()` to convert the hexadecimal representation into its corresponding sequence of bytes. It does not calculate the digest; instead, it stores a known test value so that its representation and size can be examined without using a hashing library.

Internally, SHA-256 maintains its hash value as eight 32-bit words. After all message blocks have been processed, these words are concatenated in order to form the final 256-bit digest. The total size is therefore:

$$
8 \times 32 = 256\text{ bits}
$$

This connects the final digest representation to the intermediate data structures used by the algorithm.

Therefore, a `bytes` object is suitable for representing the complete digest, a hexadecimal string is useful for displaying it, and eight 32-bit words describe its structure within the SHA-256 algorithm.

### Big-endian integers

Big-endian is a byte-order convention in which the most significant byte is stored or interpreted first. In other words, the byte containing the highest-order bits appears before the bytes containing the lower-order bits.

SHA-256 uses big-endian byte ordering when interpreting message data as 32-bit words. According to [NIST FIPS 180-4, Section 5.2.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=17), each 512-bit message block is divided into sixteen 32-bit words. Each word is formed by interpreting its four bytes in big-endian order.

For example, consider the following four bytes:
$$
12, 34, 56, 78
$$

When interpreted as a big-endian integer, the first byte (0x12) is the most significant byte, producing the integer 0x12345678. The corresponding calculation is:

$$
(0x12 \times 2^{24}) +
(0x34 \times 2^{16}) +
(0x56 \times 2^8) +
0x78
$$

In Python, the `int.from_bytes()` method can convert a sequence of bytes into an integer using a specified byte order. This is appropriate for SHA-256 because it allows message bytes to be interpreted according to the algorithm's required big-endian convention.

For example:

In [2]:
# Four bytes representing a 32-bit word
data = bytes.fromhex("12345678")

# Interpret the bytes as a big-endian integer
big_endian_value = int.from_bytes(data, byteorder="big")

# Interpret the same bytes as a little-endian integer
little_endian_value = int.from_bytes(data, byteorder="little")

print("Big-endian:", f"{big_endian_value:08x}")
print("Little-endian:", f"{little_endian_value:08x}")

Big-endian: 12345678
Little-endian: 78563412


The two results differ because the same sequence of bytes is interpreted in a different order. In big-endian representation, the first byte is the most significant; in little-endian representation, the first byte is the least significant.

Using `int.from_bytes(data, byteorder="big")` is therefore suitable when converting message data into the 32-bit words required by SHA-256. Explicitly specifying the byte order also makes the code easier to understand and avoids relying on the computer's native byte order.

## Problem 2: SHA-256 Bitwise Operations

This problem involves: implementing the Boolean and bitwise functions used by SHA-256

### 1. `Ch(x, y, z)`: Choose Function

The SHA-256 Choose function operates on three 32-bit words, x, y, and z.

It is defined as:

$$
\operatorname{Ch}(x,y,z) = (x \land y) \oplus (\neg x \land z)
$$

Where:

$(\land)$ represents bitwise AND.

$(\oplus)$ represents bitwise XOR.

$(\neg)$ represents bitwise NOT.

For each bit position, the function selects the corresponding bit from y if the bit in x is 1; otherwise, it selects the bit from z.

In SHA-256, these inputs are 32-bit words. NumPy's `uint32` type ensures that the values are represented as 32-bit unsigned integers.

Reference: [NIST FIPS 180-4, Section 4.1.2](https://doi.org/10.6028/NIST.FIPS.180-4), which defines the SHA-256 logical functions.

#### IMPLEMENTATION OF SHA-256 CHOOSE FUNCTION

In [ ]:
import numpy as np


def Ch(x: np.uint32, y: np.uint32, z: np.uint32) -> np.uint32:
    """Return the SHA-256 Choose function for three 32-bit words."""
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)

    mask = np.uint32(0xFFFFFFFF)
    return np.uint32(
        ((x & y) ^ ((~x & mask) & z))
    )

Python and NumPy bitwise NOT can produce different-looking results depending on the integer type. The mask `0xFFFFFFFF` ensures that the complement of x is restricted to 32 bits.

#### Testing the Choose Function

The following tests check the function using values that make its expected behaviour easy to verify:

1. If `x` contains only zeros, the result should equal `z`.
2. If `x` contains only ones, the result should equal `y`.
3. If `y` and `z` are identical, the result should equal their shared value, regardless of `x`.
4. The result must be a NumPy 32-bit unsigned integer.

#### Unit Tests

In [3]:

# Test 1: All bits of x are zero, so choose z.
assert Ch(
    np.uint32(0x00000000),
    np.uint32(0x12345678),
    np.uint32(0xABCDEF01),
) == np.uint32(0xABCDEF01)

# Test 2: All bits of x are one, so choose y.
assert Ch(
    np.uint32(0xFFFFFFFF),
    np.uint32(0x12345678),
    np.uint32(0xABCDEF01),
) == np.uint32(0x12345678)

# Test 3: Identical y and z must produce that value.
assert Ch(
    np.uint32(0xA5A5A5A5),
    np.uint32(0x12345678),
    np.uint32(0x12345678),
) == np.uint32(0x12345678)

# Test 4: Verify the result type.
result = Ch(
    np.uint32(0xAAAAAAAA),
    np.uint32(0xCCCCCCCC),
    np.uint32(0x55555555),
)
assert isinstance(result, np.uint32)

print("All Ch function tests passed.")


All Ch function tests passed.


### `Maj(x, y, z)`: Majority Function

The SHA-256 Majority function operates on three 32-bit words, x, y, and z.

It is defined as:

$$
\operatorname{Maj}(x,y,z) = (x \land y) \oplus (x \land z) \oplus (y \land z)
$$

Where:

$(\land)$ represents bitwise AND.

$(\oplus)$ represents bitwise XOR.

For each bit position, the function returns 1 if at least two of the three input bits are 1; otherwise, it returns 0.

NumPy's `uint32` type ensures that the values are represented as 32-bit unsigned integers.

Reference: [NIST FIPS 180-4, Section 4.1.2](https://doi.org/10.6028/NIST.FIPS.180-4), which defines the SHA-256 logical functions.

#### IMPLEMENTATION OF SHA-256 MAJORITY FUNCTION

In [ ]:
import numpy as np


def Maj(x: np.uint32, y: np.uint32, z: np.uint32) -> np.uint32:
    """Return the SHA-256 Majority function for three 32-bit words."""
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)

    return np.uint32((x & y) ^ (x & z) ^ (y & z))


#### Testing the Majority Function
The following tests verify the expected behaviour:

1. If all inputs are zero, the result should be zero.

2. If all inputs are one, the result should be one.

3. If two inputs are identical and the third is their bitwise complement, the result should equal the repeated input.

4. The result must be a NumPy 32-bit unsigned integer.

#### Unit Tests

In [6]:

# Test 1: All inputs are zero.
assert Maj(
    np.uint32(0x00000000),
    np.uint32(0x00000000),
    np.uint32(0x00000000),
) == np.uint32(0x00000000)

# Test 2: All inputs are one.
assert Maj(
    np.uint32(0xFFFFFFFF),
    np.uint32(0xFFFFFFFF),
    np.uint32(0xFFFFFFFF),
) == np.uint32(0xFFFFFFFF)

# Test 3: Two inputs match, so their value is selected.
assert Maj(
    np.uint32(0xAAAAAAAA),
    np.uint32(0xAAAAAAAA),
    np.uint32(0x55555555),
) == np.uint32(0xAAAAAAAA)

# Test 4: Verify the result type.
result = Maj(
    np.uint32(0xAAAAAAAA),
    np.uint32(0xCCCCCCCC),
    np.uint32(0xF0F0F0F0),
)
assert isinstance(result, np.uint32)

print("All Maj function tests passed.")


All Maj function tests passed.


### `Sigma0(x)`: Uppercase Sigma Zero Function

The SHA-256 function $(\Sigma_0(x))$ operates on a 32-bit word and is defined as:

$$
\Sigma_0(x) = \operatorname{ROTR}^{2}(x)
\oplus \operatorname{ROTR}^{13}(x)
\oplus \operatorname{ROTR}^{22}(x)
$$

Where:

$(\operatorname{ROTR}^{n}(x))$ rotates the bits of x to the right by n positions.

$(\oplus)$ represents bitwise XOR.

Rotation moves the bits that leave the right-hand side back to the left-hand side, unlike a normal right shift.

Reference: [NIST FIPS 180-4, Section 4.1.2](https://doi.org/10.6028/NIST.FIPS.180-4).

In [8]:

import numpy as np


def _rotr(x: np.uint32, n: int) -> np.uint32:
    """Rotate a 32-bit unsigned integer right by n bits."""
    x = np.uint32(x)
    return np.uint32((x >> n) | (x << (32 - n)))


def Sigma0(x: np.uint32) -> np.uint32:
    """Return the SHA-256 uppercase Sigma zero function."""
    x = np.uint32(x)
    return np.uint32(
        _rotr(x, 2) ^ _rotr(x, 13) ^ _rotr(x, 22)
    )


#### Testing the Sigma0 Function

The following tests verify that the function behaves correctly:

1. An input of zero should produce zero.

2. An input of all ones should produce all ones.

3. An input of 0x00000001 should produce the expected result after the specified rotations and XOR operations.

4. The output should be a NumPy 32-bit unsigned integer (np.uint32).

#### Unit Tests

In [9]:

# Test 1: Zero input.
assert Sigma0(np.uint32(0x00000000)) == np.uint32(0x00000000)

# Test 2: All bits set.
assert Sigma0(np.uint32(0xFFFFFFFF)) == np.uint32(0xFFFFFFFF)

# Test 3: Verify rotations for an input of 1.
assert Sigma0(np.uint32(0x00000001)) == np.uint32(0x40080400)

# Test 4: Verify the result type.
assert isinstance(Sigma0(np.uint32(0x12345678)), np.uint32)

print("All Sigma0 function tests passed.")


All Sigma0 function tests passed.


### `Sigma1(x)`: Uppercase Sigma One Function

The SHA-256 function $\Sigma_1(x)$ operates on a 32-bit word and is defined as:

$$
\Sigma_1(x) = \operatorname{ROTR}^{6}(x)
\oplus \operatorname{ROTR}^{11}(x)
\oplus \operatorname{ROTR}^{25}(x)
$$

Where:

$(\operatorname{ROTR}^{n}(x))$ rotates the bits of x to the right by n positions.

$(\oplus)$ represents bitwise XOR.

Rotation moves the bits that leave the right-hand side back to the left-hand side, unlike a normal right shift.

The function combines three right rotations using XOR to produce a new 32-bit word.

NumPy's `uint32` type ensures that the values are represented as 32-bit unsigned integers.

Reference: [NIST FIPS 180-4, Section 4.1.2](https://doi.org/10.6028/NIST.FIPS.180-4).

#### IMPLEMENTATION OF SHA-256 UPPERCASE SIGMA ONE FUNCTION

In [ ]:
def Sigma1(x: np.uint32) -> np.uint32:
    """Return the SHA-256 uppercase Sigma one function."""
    x = np.uint32(x)

    return np.uint32(
        _rotr(x, 6) ^ _rotr(x, 11) ^ _rotr(x, 25)
    )

#### Testing the Sigma1 Function

The following tests verify that the function behaves correctly:

1. An input of zero should produce zero.

2. An input of all ones should produce all ones.

3. An input of 0x00000001 should produce the expected result after the specified rotations and XOR operations.

4. The output should be a NumPy 32-bit unsigned integer (np.uint32).

#### Unit Tests

In [ ]:
# Test 1: Zero input.
assert Sigma1(np.uint32(0x00000000)) == np.uint32(0x00000000)

# Test 2: All bits set.
assert Sigma1(np.uint32(0xFFFFFFFF)) == np.uint32(0xFFFFFFFF)

# Test 3: Verify rotations for an input of 1.
assert Sigma1(np.uint32(0x00000001)) == np.uint32(0x04200080)

# Test 4: Verify the result type.
assert isinstance(Sigma1(np.uint32(0x12345678)), np.uint32)

print("All Sigma1 function tests passed.")

### `sigma0(x)`: Lowercase Sigma Zero Function

The SHA-256 function $\sigma_0(x)$ operates on a 32-bit word and is defined as:

$$
\sigma_0(x) = \operatorname{ROTR}^{7}(x)
\oplus \operatorname{ROTR}^{18}(x)
\oplus \operatorname{SHR}^{3}(x)
$$

Where:

$(\operatorname{ROTR}^{n}(x))$ rotates the bits of x to the right by n positions.

$(\operatorname{SHR}^{n}(x))$ shifts the bits of x to the right by n positions, filling the empty positions with zeros.

$(\oplus)$ represents bitwise XOR.

Unlike a right rotation, a right shift discards the bits that leave the right-hand side. The function combines two rotations and one shift using XOR.

NumPy's `uint32` type ensures that the values are represented as 32-bit unsigned integers.

Reference: [NIST FIPS 180-4, Section 4.1.2](https://doi.org/10.6028/NIST.FIPS.180-4).

#### IMPLEMENTATION OF SHA-256 LOWERCASE SIGMA ZERO FUNCTION

In [ ]:
def sigma0(x: np.uint32) -> np.uint32:
    """Return the SHA-256 lowercase Sigma zero function."""
    x = np.uint32(x)

    return np.uint32(
        _rotr(x, 7) ^ _rotr(x, 18) ^ (x >> 3)
    )

#### Testing the sigma0 Function

The following tests verify that the function behaves correctly:

1. An input of zero should produce zero.

2. An input of all ones should produce the expected result after the rotations, shift and XOR operations.

3. An input of 0x00000001 should produce the expected result after the specified rotations, shift and XOR operations.

4. The output should be a NumPy 32-bit unsigned integer (np.uint32).

#### Unit Tests

In [ ]:
# Test 1: Zero input.
assert sigma0(np.uint32(0x00000000)) == np.uint32(0x00000000)

# Test 2: All bits set.
assert sigma0(np.uint32(0xFFFFFFFF)) == np.uint32(0x1FFFFFFF)

# Test 3: Verify rotations and shift for an input of 1.
assert sigma0(np.uint32(0x00000001)) == np.uint32(0x02004000)

# Test 4: Verify the result type.
assert isinstance(sigma0(np.uint32(0x12345678)), np.uint32)

print("All sigma0 function tests passed.")

### `sigma1(x)`: Lowercase Sigma One Function

The SHA-256 function $\sigma_1(x)$ operates on a 32-bit word and is defined as:

$$
\sigma_1(x) = \operatorname{ROTR}^{17}(x)
\oplus \operatorname{ROTR}^{19}(x)
\oplus \operatorname{SHR}^{10}(x)
$$

Where:

$(\operatorname{ROTR}^{n}(x))$ rotates the bits of x to the right by n positions.

$(\operatorname{SHR}^{n}(x))$ shifts the bits of x to the right by n positions, filling the empty positions with zeros.

$(\oplus)$ represents bitwise XOR.

Unlike a right rotation, a right shift discards the bits that leave the right-hand side.

The function combines two right rotations and one logical right shift using XOR to produce a new 32-bit word.

NumPy's `uint32` type ensures that the values are represented as 32-bit unsigned integers.

Reference: [NIST FIPS 180-4, Section 4.1.2](https://doi.org/10.6028/NIST.FIPS.180-4).

#### IMPLEMENTATION OF SHA-256 LOWERCASE SIGMA ONE FUNCTION

In [ ]:
def sigma1(x: np.uint32) -> np.uint32:
    """Return the SHA-256 lowercase Sigma one function."""
    x = np.uint32(x)

    return np.uint32(
        _rotr(x, 17) ^ _rotr(x, 19) ^ (x >> 10)
    )

#### Testing the sigma1 Function

The following tests verify that the function behaves correctly:

1. An input of zero should produce zero.

2. An input of all ones should produce the expected result after the rotations, shift and XOR operations.

3. An input of 0x00000001 should produce the expected result after the specified rotations, shift and XOR operations.

4. The output should be a NumPy 32-bit unsigned integer (np.uint32).

#### Unit Tests

In [ ]:
# Test 1: Zero input.
assert sigma1(np.uint32(0x00000000)) == np.uint32(0x00000000)

# Test 2: All bits set.
assert sigma1(np.uint32(0xFFFFFFFF)) == np.uint32(0x003FFFFF)

# Test 3: Verify rotations and shift for an input of 1.
assert sigma1(np.uint32(0x00000001)) == np.uint32(0x0000A000)

# Test 4: Verify the result type.
assert isinstance(sigma1(np.uint32(0x12345678)), np.uint32)

print("All sigma1 function tests passed.")

## Problem 3: Generating the SHA-256 Constants

This problem involves: using Python and NumPy to generate the 64 SHA-256 round constants from the fractional parts of the cube roots of the first 64 prime numbers, as defined in the Secure Hash Standard.

Reference: [NIST FIPS 180-4, Section 4.2.2, page 11](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19).

### 1. `primes(n)`: Generate Prime Numbers

A prime number is a positive integer greater than 1 that is divisible only by 1 and itself.

The function `primes(n)` generates the first n prime numbers in ascending order.

To determine whether a number is prime, the function checks whether it is divisible by any integer between 2 and its square root.

If a number has a factor greater than its square root, it must also have a corresponding factor smaller than its square root. Therefore, checking beyond the square root is unnecessary.

The first five prime numbers are 2, 3, 5, 7 and 11.

These prime numbers will later be used to generate the SHA-256 round constants from the fractional parts of their cube roots.

Reference: [NIST FIPS 180-4, Section 4.2.2, page 11](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19).

#### IMPLEMENTATION OF PRIME NUMBER GENERATION

In [8]:
def primes(n: int) -> list[int]:
    """Return the first n prime numbers in ascending order."""

    # Reject negative inputs.
    if n < 0:
        raise ValueError("n must be non-negative")

    # Store discovered primes, starting with the first possible prime.
    result = []
    candidate = 2

    # Continue checking numbers until n primes have been found.
    while len(result) < n:
        is_prime = True

        # A composite number must have a divisor <= its square root.
        for divisor in range(2, int(candidate ** 0.5) + 1):
            if candidate % divisor == 0:
                is_prime = False
                break

        # Add the candidate only if no divisor was found.
        if is_prime:
            result.append(candidate)

        candidate += 1

    return result

#### Testing the primes Function

The following tests verify that the function behaves correctly:

1. An input of zero should produce an empty list.

2. An input of one should produce a list containing only the first prime number, 2.

3. An input of five should produce the first five prime numbers in ascending order.

4. An input of 64 should produce exactly 64 prime numbers, with 311 as the final value.

5. A negative input should raise a ValueError.

6. The generated prime numbers should be unique and arranged in ascending order.

#### References

The expected prime number values used in these tests were verified against the [Online Encyclopedia of Integer Sequences (OEIS), A000040](https://oeis.org/A000040).

The requirement to generate the first 64 prime numbers is based on [NIST FIPS 180-4, Section 4.2.2, page 11](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19).

#### Unit Tests

In [9]:
# Test 1: Verify that requesting zero primes returns an empty list.
assert primes(0) == []

# Test 2: Verify that the first prime number is 2.
assert primes(1) == [2]

# Test 3: Verify the first five prime numbers.
assert primes(5) == [2, 3, 5, 7, 11]

# Test 4: Verify that 64 primes are generated and the last is 311.
result = primes(64)
assert len(result) == 64
assert result[-1] == 311

# Test 5: Verify that a negative input raises a ValueError.
try:
    primes(-1)
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for negative input")

# Test 6: Verify that the generated primes are unique and ordered.
assert result == sorted(result)
assert len(result) == len(set(result))

print("All primes function tests passed.")

All primes function tests passed.


### 2. Calculating the Cube Roots of the First 64 Primes

The SHA-256 algorithm uses the first 64 prime numbers to generate its round constants.

Each constant is derived from the fractional part of the cube root of a prime number.

The cube root of a number p is defined as:

$$
r = \sqrt[3]{p} = p^{1/3}
$$

Where:

$p$ represents a prime number.

$r$ represents the cube root of that prime number.

The function `cube_roots(n)` uses the previously implemented `primes(n)` function to generate the first n primes, then calculates their cube roots using NumPy's `cbrt()` function.

NumPy's `float64` type is used to represent the cube roots because they contain fractional values. These fractional parts will later be converted into 32-bit unsigned integers to generate the SHA-256 round constants.

Reference: [NIST FIPS 180-4, Section 4.2.2, page 11](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19).

Reference: [NumPy documentation — numpy.cbrt](https://numpy.org/doc/stable/reference/generated/numpy.cbrt.html).

#### Implementation of Cube Root Calculation

In [10]:
import numpy as np


def cube_roots(n: int) -> np.ndarray:
    """Return the cube roots of the first n prime numbers."""

    # Generate the first n prime numbers.
    prime_numbers = primes(n)

    # Convert primes to float64 for fractional calculations.
    prime_array = np.array(prime_numbers, dtype=np.float64)

    # Calculate the cube root of each prime number.
    return np.cbrt(prime_array)

#### Testing the cube_roots Function

The following tests verify that the function behaves correctly:

1. An input of zero should produce an empty NumPy array.

2. An input of one should produce the cube root of the first prime number, 2.

3. An input of five should produce the cube roots of the first five primes.

4. An input of 64 should produce exactly 64 cube roots, with the final value corresponding to the cube root of 311.

5. The output should be a NumPy array containing 64-bit floating-point values.

Floating-point calculations can introduce small rounding differences. Therefore, `np.isclose()` and `np.allclose()` are used to compare approximate numerical values rather than requiring exact equality.

Reference: [NumPy documentation — numpy.isclose](https://numpy.org/doc/stable/reference/generated/numpy.isclose.html).

#### Unit Tests

In [11]:
# Test 1: Verify that zero primes produces an empty array.
assert cube_roots(0).size == 0

# Test 2: Verify the cube root of the first prime number.
assert np.isclose(cube_roots(1)[0], 2 ** (1 / 3))

# Test 3: Verify the cube roots of the first five primes.
expected = np.array([2, 3, 5, 7, 11], dtype=np.float64)
assert np.allclose(cube_roots(5) ** 3, expected)

# Test 4: Verify that 64 cube roots are generated.
result = cube_roots(64)
assert len(result) == 64
assert np.isclose(result[-1] ** 3, 311)

# Test 5: Verify the output type and precision.
assert isinstance(result, np.ndarray)
assert result.dtype == np.float64

print("All cube_roots function tests passed.")

All cube_roots function tests passed.


### 3. Extracting the Fractional Parts of the Cube Roots

The SHA-256 round constants are derived from the fractional parts of the cube roots of the first 64 prime numbers.

The fractional part of a positive number is obtained by subtracting its integer part.

It is defined as:

$$
f = r - \lfloor r \rfloor
$$

Where:

$r$ represents the cube root of a prime number.

$\lfloor r \rfloor$ represents the greatest integer less than or equal to $r$.

$f$ represents the fractional part of the cube root.

For example, the cube root of 2 is approximately 1.259921, so its fractional part is approximately 0.259921.

The function `fractional_parts(n)` uses the previously implemented `cube_roots(n)` function and NumPy's `modf()` function to extract the fractional part of each cube root.

NumPy's `float64` type is used to preserve floating-point precision. The resulting fractional values will later be converted into 32-bit unsigned integers to generate the SHA-256 round constants.

Reference: [NIST FIPS 180-4, Section 4.2.2, page 11](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19).

Reference: [NumPy documentation — numpy.modf](https://numpy.org/doc/stable/reference/generated/numpy.modf.html).

#### Implementation of Fractional Part Extraction

In [12]:
def fractional_parts(n: int) -> np.ndarray:
    """Return the fractional parts of the first n prime cube roots."""

    # Calculate the cube roots of the first n primes.
    roots = cube_roots(n)

    # Separate the fractional and integer parts of each cube root.
    fractions, _ = np.modf(roots)

    # Return only the fractional parts.
    return fractions

#### Testing the fractional_parts Function

The following tests verify that the function behaves correctly:

1. An input of zero should produce an empty NumPy array.

2. An input of one should produce the fractional part of the cube root of 2.

3. An input of five should produce fractional values corresponding to the first five prime cube roots.

4. An input of 64 should produce exactly 64 fractional values, each greater than or equal to zero and less than one.

5. The output should be a NumPy array containing 64-bit floating-point values.

Floating-point calculations may introduce small rounding differences. Therefore, `np.isclose()` and `np.allclose()` are used when comparing calculated fractional values.

Reference: [NumPy documentation — numpy.isclose](https://numpy.org/doc/stable/reference/generated/numpy.isclose.html).

#### Unit Tests

In [13]:
# Test 1: Verify that zero primes produces an empty array.
assert fractional_parts(0).size == 0

# Test 2: Verify the fractional part of the cube root of 2.
assert np.isclose(fractional_parts(1)[0], np.cbrt(2) - 1)

# Test 3: Verify the fractional parts of the first five cube roots.
result_five = fractional_parts(5)
expected = cube_roots(5) - np.floor(cube_roots(5))
assert np.allclose(result_five, expected)

# Test 4: Verify that 64 fractional values are generated within [0, 1).
result = fractional_parts(64)
assert len(result) == 64
assert np.all((result >= 0) & (result < 1))

# Test 5: Verify the output type and precision.
assert isinstance(result, np.ndarray)
assert result.dtype == np.float64

print("All fractional_parts function tests passed.")

All fractional_parts function tests passed.


### 4. Converting Fractional Parts to 32-Bit Unsigned Integers

The SHA-256 round constants are 32-bit unsigned integers derived from the fractional parts of the cube roots of the first 64 prime numbers.

Each fractional value is multiplied by $2^{32}$, and the integer portion of the result is extracted.

The conversion is defined as:

$$
K_i = \left\lfloor 2^{32} \times f_i \right\rfloor
$$

Where:

$K_i$ represents the resulting 32-bit SHA-256 round constant.

$f_i$ represents the fractional part of the cube root of the corresponding prime number.

$2^{32}$ scales the fractional value into the range of a 32-bit unsigned integer.

$\lfloor \cdot \rfloor$ represents the floor operation, which discards the fractional component.

The function `round_constants(n)` uses the previously implemented `fractional_parts(n)` function to obtain the fractional values.

NumPy's `uint32` type is then used to represent the resulting constants as 32-bit unsigned integers.

For example, the first SHA-256 round constant is derived from the fractional part of the cube root of 2, producing `0x428A2F98`.

Reference: [NIST FIPS 180-4, Section 4.2.2, page 11](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19).

Reference: [NumPy documentation — numpy.uint32](https://numpy.org/doc/stable/reference/arrays.scalars.html).

#### Implementation of 32-Bit Unsigned Integer Conversion

In [1]:
def round_constants(n: int) -> np.ndarray:
    """Generate SHA-256 round constants as 32-bit unsigned integers."""

    # Extract the fractional parts of the first n prime cube roots.
    fractions = fractional_parts(n)

    # Scale each fractional value to the 32-bit unsigned integer range.
    scaled = fractions * (2 ** 32)

    # Discard the remaining fractional component.
    integers = np.floor(scaled)

    # Convert the resulting values to 32-bit unsigned integers.
    return integers.astype(np.uint32)

#### Testing the round_constants Function

The following tests verify that the function behaves correctly:

1. An input of zero should produce an empty NumPy array.

2. An input of one should produce the first SHA-256 round constant, `0x428A2F98`.

3. An input of four should produce the first four SHA-256 round constants specified by NIST.

4. An input of 64 should produce exactly 64 round constants, with the final value equal to `0xC67178F2`.

5. The output should be a NumPy array containing 32-bit unsigned integers (`np.uint32`).

The expected hexadecimal values are taken from NIST FIPS 180-4, Section 4.2.2.

Unlike the previous floating-point tests, these tests use exact equality because SHA-256 round constants must match the values specified in the standard.

Reference: [NIST FIPS 180-4, Section 4.2.2, page 11](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19).

#### Unit Tests

In [2]:
# Test 1: Verify that zero primes produces an empty array.
assert round_constants(0).size == 0

# Test 2: Verify the first SHA-256 round constant.
assert round_constants(1)[0] == np.uint32(0x428A2F98)

# Test 3: Verify the first four constants against NIST.
expected = np.array([
    0x428A2F98,
    0x71374491,
    0xB5C0FBCF,
    0xE9B5DBA5
], dtype=np.uint32)

assert np.array_equal(round_constants(4), expected)

# Test 4: Verify that 64 constants are generated.
result = round_constants(64)
assert len(result) == 64
assert result[-1] == np.uint32(0xC67178F2)

# Test 5: Verify the output type and precision.
assert isinstance(result, np.ndarray)
assert result.dtype == np.uint32

print("All round_constants function tests passed.")

NameError: name 'fractional_parts' is not defined

### 5. Displaying the SHA-256 Round Constants in Hexadecimal

The SHA-256 round constants are represented as 32-bit unsigned integers and are conventionally displayed in hexadecimal notation.

Hexadecimal is a base-16 number system that uses the digits 0–9 and the letters A–F.

Each hexadecimal digit represents four binary bits. Therefore, a 32-bit unsigned integer can be represented using exactly eight hexadecimal digits.

The function `display_constants(n)` uses the previously implemented `round_constants(n)` function to generate the constants and display them in hexadecimal.

Python's hexadecimal formatting syntax `08X` ensures that:

1. Each value is displayed using at least eight digits.
2. Leading zeros are included where necessary.
3. Hexadecimal letters are displayed in uppercase.

The `0x` prefix is included to identify each value as hexadecimal.

For example, the first SHA-256 round constant is represented as `0x428A2F98`.

The hexadecimal representation allows the generated constants to be compared directly against the values published in the Secure Hash Standard.

Reference: [NIST FIPS 180-4, Section 4.2.2, page 11](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19).

Reference: [Python documentation — Format Specification Mini-Language](https://docs.python.org/3/library/string.html#format-specification-mini-language).

#### Implementation of Hexadecimal Display

In [ ]:
def display_constants(n: int) -> list[str]:
    """Display the first n SHA-256 round constants in hexadecimal."""

    # Generate the SHA-256 round constants as uint32 values.
    constants = round_constants(n)

    # Convert each constant to an eight-digit hexadecimal string.
    hexadecimal = [f"0x{int(value):08X}" for value in constants]

    # Display each constant alongside its index.
    for index, value in enumerate(hexadecimal):
        print(f"K[{index:02d}] = {value}")

    return hexadecimal


# Display all 64 SHA-256 round constants.
hex_constants = display_constants(64)

#### Testing the display_constants Function

The following tests verify that the function behaves correctly:

1. An input of zero should produce an empty list.

2. An input of one should produce the first SHA-256 round constant in hexadecimal notation.

3. An input of four should produce the first four constants in the correct order.

4. An input of 64 should produce exactly 64 hexadecimal constants, with the final value equal to `0xC67178F2`.

5. Each hexadecimal value should be a string containing the `0x` prefix followed by exactly eight hexadecimal digits.

The expected values are taken from NIST FIPS 180-4, Section 4.2.2.

Reference: [NIST FIPS 180-4, Section 4.2.2, page 11](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19).

#### Unit Tests

In [ ]:
# Test 1: Verify that zero constants produces an empty list.
assert display_constants(0) == []

# Test 2: Verify the hexadecimal representation of the first constant.
assert hex_constants[0] == "0x428A2F98"

# Test 3: Verify the first four hexadecimal constants.
expected = [
    "0x428A2F98",
    "0x71374491",
    "0xB5C0FBCF",
    "0xE9B5DBA5"
]

assert hex_constants[:4] == expected

# Test 4: Verify that 64 hexadecimal constants are generated.
assert len(hex_constants) == 64
assert hex_constants[-1] == "0xC67178F2"

# Test 5: Verify the hexadecimal format of every constant.
assert all(
    isinstance(value, str)
    and value.startswith("0x")
    and len(value) == 10
    and all(char in "0123456789ABCDEF" for char in value[2:])
    for value in hex_constants
)

print("All display_constants function tests passed.")

### 6. Comparing Generated Constants with the Secure Hash Standard

The Secure Hash Standard (NIST FIPS 180-4) specifies 64 round constants for SHA-256.

These constants are derived from the fractional parts of the cube roots of the first 64 prime numbers and are represented as 32-bit unsigned integers.

To verify the implementation, the generated constants are compared against the hexadecimal values published in the standard.

The comparison involves:

1. Storing the 64 reference constants from NIST in a NumPy array of type `uint32`.
2. Generating the constants using the previously implemented `round_constants(64)` function.
3. Comparing each generated constant against its corresponding reference value.
4. Reporting whether all 64 constants match exactly.

NumPy's `array_equal()` function is used because SHA-256 requires exact 32-bit values rather than approximate floating-point equality.

Any differences may indicate an error in the implementation or a floating-point precision issue during cube root calculation.

Reference: [NIST FIPS 180-4, Section 4.2.2, page 11](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19).

Reference: [NumPy documentation — numpy.array_equal](https://numpy.org/doc/stable/reference/generated/numpy.array_equal.html).

#### Implementation of SHA-256 Constant Verification

In [ ]:
# Store the 64 SHA-256 round constants specified by NIST.
nist_constants = np.array([
    0x428A2F98, 0x71374491, 0xB5C0FBCF, 0xE9B5DBA5,
    0x3956C25B, 0x59F111F1, 0x923F82A4, 0xAB1C5ED5,
    0xD807AA98, 0x12835B01, 0x243185BE, 0x550C7DC3,
    0x72BE5D74, 0x80DEB1FE, 0x9BDC06A7, 0xC19BF174,
    0xE49B69C1, 0xEFBE4786, 0x0FC19DC6, 0x240CA1CC,
    0x2DE92C6F, 0x4A7484AA, 0x5CB0A9DC, 0x76F988DA,
    0x983E5152, 0xA831C66D, 0xB00327C8, 0xBF597FC7,
    0xC6E00BF3, 0xD5A79147, 0x06CA6351, 0x14292967,
    0x27B70A85, 0x2E1B2138, 0x4D2C6DFC, 0x53380D13,
    0x650A7354, 0x766A0ABB, 0x81C2C92E, 0x92722C85,
    0xA2BFE8A1, 0xA81A664B, 0xC24B8B70, 0xC76C51A3,
    0xD192E819, 0xD6990624, 0xF40E3585, 0x106AA070,
    0x19A4C116, 0x1E376C08, 0x2748774C, 0x34B0BCB5,
    0x391C0CB3, 0x4ED8AA4A, 0x5B9CCA4F, 0x682E6FF3,
    0x748F82EE, 0x78A5636F, 0x84C87814, 0x8CC70208,
    0x90BEFFFA, 0xA4506CEB, 0xBEF9A3F7, 0xC67178F2
], dtype=np.uint32)


def verify_constants() -> bool:
    """Compare generated SHA-256 constants against the NIST standard."""

    # Generate the 64 round constants using the previous implementation.
    generated = round_constants(64)

    # Compare each generated constant with its NIST reference value.
    matches = np.array_equal(generated, nist_constants)

    # Report any differences using hexadecimal notation.
    if not matches:
        for index, (actual, expected) in enumerate(
            zip(generated, nist_constants)
        ):
            if actual != expected:
                print(
                    f"K[{index:02d}]: Generated 0x{int(actual):08X}, "
                    f"Expected 0x{int(expected):08X}"
                )

    return bool(matches)


# Verify the generated constants.
if verify_constants():
    print("All 64 SHA-256 constants match the NIST standard.")
else:
    print("Some SHA-256 constants do not match the NIST standard.")

#### Testing the Constant Verification

The following tests verify that the comparison works correctly:

1. The NIST reference array should contain exactly 64 constants.

2. The reference constants should be represented as NumPy 32-bit unsigned integers.

3. The first and last reference constants should match the values specified by NIST.

4. The generated constants should match all 64 reference values exactly.

5. The verification function should return a Boolean result indicating whether the constants match.

These tests use exact equality because even a one-bit difference would produce an incorrect SHA-256 round constant.

Reference: [NIST FIPS 180-4, Section 4.2.2, page 11](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19).

#### Unit Tests

In [ ]:
# Test 1: Verify that NIST specifies exactly 64 round constants.
assert len(nist_constants) == 64

# Test 2: Verify the reference constants use 32-bit unsigned integers.
assert nist_constants.dtype == np.uint32

# Test 3: Verify the first and last NIST constants.
assert nist_constants[0] == np.uint32(0x428A2F98)
assert nist_constants[-1] == np.uint32(0xC67178F2)

# Test 4: Verify that all generated constants match the NIST values.
assert np.array_equal(round_constants(64), nist_constants)

# Test 5: Verify that the comparison returns a Boolean result.
assert isinstance(verify_constants(), bool)

print("All SHA-256 constant verification tests passed.")

#### Conclusion

This problem demonstrates how the SHA-256 round constants are derived from the first 64 prime numbers.

The implementation generates the prime numbers, calculates their cube roots, extracts the fractional parts, and converts the results into 32-bit unsigned integers.

The generated constants are then displayed in hexadecimal and compared against the 64 reference values published in NIST FIPS 180-4.

An exact match confirms that the generated constants agree with the Secure Hash Standard. Any differences must be investigated, particularly those caused by floating-point precision during the cube root calculations.

## Problem 4: Padding and Parsing Messages

This problem involves: 

## Problem 5: The SHA-256 Compression Function

This problem involves: 

## Problem 6: Complete SHA-256

This problem involves: 

## End